# Pinecone Vector DB: Cloud Serverless Vector Search & RAG Architecture

## 1. Overview and Core Motivation
**Pinecone** is a purpose-built, cloud-native vector database engineered for enterprise RAG, semantic recommendation engines, and real-time similarity search:
- **Serverless Decoupled Architecture**: Automatically scales compute and storage independently, eliminating pod provisioning, cluster sizing, and manual sharding.
- **Sub-50ms Global Query Latency**: Powered by proprietary approximate nearest neighbor (ANN) search algorithms optimized for cloud blob storage and vector caching.
- **Single-Stage Metadata Filtering**: Combines inverted metadata indices directly with vector graph traversal so that filtered searches incur zero latency penalties.
- **Multi-Tenant Namespaces**: Partition a single index into logically isolated partitions for different tenants, users, or document corpora.

---

## 2. Serverless Storage & Search Pipeline

```
┌────────────────────────────────────────────────────────────────────────────────────────┐
│                       PINECONE SERVERLESS CLOUD ARCHITECTURE                           │
│                                                                                        │
│     Write Pipeline                                   Read / Search Pipeline            │
│  ┌──────────────────┐                             ┌───────────────────────────────┐    │
│  │ Document Chunks  │                             │ User Query + Metadata Filters │    │
│  │ & 384d Vectors   │                             │  {"source": "tweet"}          │    │
│  └────────┬─────────┘                             └───────────────┬───────────────┘    │
│           │                                                       │                    │
│           ▼                                                       ▼                    │
│  ┌──────────────────┐                             ┌───────────────────────────────┐    │
│  │ Stateless Writer │                             │ Stateless Query Workers       │    │
│  │  (Micro-batches) │                             │ (Horizontal Auto-scaling)     │    │
│  └────────┬─────────┘                             └───────────────┬───────────────┘    │
│           │                                                       │                    │
│           ▼                                                       ▼                    │
│  ┌────────────────────────────────────────────────────────────────────────────────┐    │
│  │                    DECOUPLED CLOUD OBJECT STORAGE (S3/GCS)                     │    │
│  │                                                                                │    │
│  │  ┌─────────────────────────┐                   ┌─────────────────────────┐     │    │
│  │  │ Quantized Vector Blocks │                   │ Inverted Metadata Index │     │    │
│  │  │ (Geometric Embeddings)  │                   │ (Source, Category, User)│     │    │
│  │  └─────────────────────────┘                   └─────────────────────────┘     │    │
│  └────────────────────────────────────────────────────────────────────────────────┘    │
└────────────────────────────────────────────────────────────────────────────────────────┘
```

```mermaid
flowchart TD
    subgraph Client Application
    Doc[Raw Text Documents] --> Embed[HuggingFace all-MiniLM-L6-v2]
    Embed --> Vec[384-Dimensional Vectors]
    end

    subgraph Pinecone Serverless Cloud
    Vec --> Ingest[Ingestion Gateway API]
    Ingest --> Namespace[Target Namespace Partition]
    Namespace --> Storage[(Decoupled Blob Storage)]
    
    Q[User Question] --> QEmbed[HuggingFace embed_query]
    QEmbed --> QueryWorkers[Stateless Query Workers]
    QueryWorkers --> Filter[Inverted Metadata Pre-filter]
    Storage --> QueryWorkers
    Filter --> ANN[Approximate Nearest Neighbor Search]
    ANN --> TopK[Top-K Scored Matches]
    end

    subgraph RAG Generation
    TopK --> Prompt[LangChain QA Prompt Template]
    Prompt --> Groq[Groq LPU LLM - openai/gpt-oss-120b]
    Groq --> Answer[Grounded Natural Language Answer]
    end
```

In [ ]:
# Step 1: Install official Pinecone and LangChain integration libraries
!pip install -qU langchain langchain-core langchain-pinecone pinecone langchain-groq langchain-huggingface sentence-transformers

In [ ]:
# Step 2: Environment Setup & API Configuration
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv, find_dotenv

# Search for .env
load_dotenv('../.env')
load_dotenv(find_dotenv())

# Pinecone & Groq API Keys
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "")
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")

print("API Key Configuration Status:")
print(f"- PINECONE_API_KEY: {'CONFIGURED' if PINECONE_API_KEY and 'your' not in PINECONE_API_KEY else 'NOT CONFIGURED (Will use safe emulation fallback)'}")
print(f"- GROQ_API_KEY:     {'CONFIGURED' if GROQ_API_KEY else 'NOT CONFIGURED'}")

In [ ]:
# Step 3: Initialize HuggingFace Embeddings & Groq LLM
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain.chat_models import init_chat_model

# 1. Embeddings: HuggingFace sentence-transformers/all-MiniLM-L6-v2 (384 dimensions)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("HuggingFace all-MiniLM-L6-v2 embedding model loaded successfully (384 dimensions).")

# 2. LLM: Groq ultra-fast LPU inference
try:
    llm = init_chat_model('groq:openai/gpt-oss-120b')
    test_res = llm.invoke("Respond with 'Pinecone Ready' in 2 words.")
    print(f"Groq LLM Status: {test_res.content}")
except Exception as e:
    print(f"Fallback to groq:llama-3.3-70b-versatile due to: {e}")
    llm = init_chat_model('groq:llama-3.3-70b-versatile')

In [ ]:
# Step 4: Provision or Connect to Pinecone Serverless Index
INDEX_NAME = "rag-enterprise-index"
VECTOR_DIMENSION = 384  # Must match all-MiniLM-L6-v2 dimensions exactly!

has_pinecone_credentials = bool(PINECONE_API_KEY) and "your" not in PINECONE_API_KEY.lower()

if has_pinecone_credentials:
    try:
        from pinecone import Pinecone, ServerlessSpec
        from langchain_pinecone import PineconeVectorStore
        
        pc = Pinecone(api_key=PINECONE_API_KEY)
        
        # Check if index exists; if not, create serverless index
        if not pc.has_index(INDEX_NAME):
            print(f"Creating new serverless Pinecone index '{INDEX_NAME}' on AWS us-east-1...")
            pc.create_index(
                name=INDEX_NAME,
                dimension=VECTOR_DIMENSION,
                metric="cosine",
                spec=ServerlessSpec(cloud="aws", region="us-east-1"),
            )
        
        index = pc.Index(INDEX_NAME)
        vector_store = PineconeVectorStore(index=index, embedding=embedding_model)
        print(f"Connected live to Pinecone index: '{INDEX_NAME}'")
    except Exception as e:
        print(f"Pinecone live connection notice ({e}). Initializing transparent local vectorstore fallback.")
        from langchain_core.vectorstores import InMemoryVectorStore
        vector_store = InMemoryVectorStore(embedding=embedding_model)
else:
    print("No live Pinecone API key configured; initializing transparent local vectorstore fallback for interactive execution.")
    from langchain_core.vectorstores import InMemoryVectorStore
    vector_store = InMemoryVectorStore(embedding=embedding_model)

vector_store

In [ ]:
# Step 5: Define diverse enterprise documents with structured metadata
from langchain_core.documents import Document

documents = [
    Document(
        page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
        metadata={"source": "tweet", "category": "food", "author": "alice"},
    ),
    Document(
        page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
        metadata={"source": "news", "category": "weather", "author": "weather_service"},
    ),
    Document(
        page_content="Building an exciting new production RAG system with Pinecone and Groq - sub-50ms latency!",
        metadata={"source": "tweet", "category": "tech", "author": "dev_guru"},
    ),
    Document(
        page_content="Robbers broke into the downtown city bank and stole $1 million in cash overnight.",
        metadata={"source": "news", "category": "crime", "author": "crime_beat"},
    ),
    Document(
        page_content="Wow! That new space exploration film was an amazing movie. The visual cinematography was stunning.",
        metadata={"source": "tweet", "category": "entertainment", "author": "movie_buff"},
    ),
    Document(
        page_content="Is the new flagship smartphone worth the price? Read this full review of display and battery endurance.",
        metadata={"source": "website", "category": "tech", "author": "tech_radar"},
    ),
    Document(
        page_content="The top 10 soccer players in the world right now ranked by career milestones and trophies.",
        metadata={"source": "website", "category": "sports", "author": "world_sports"},
    ),
    Document(
        page_content="LangGraph and Pinecone Serverless enable stateful, multi-tenant agentic workflows at scale.",
        metadata={"source": "tweet", "category": "tech", "author": "ai_architect"},
    ),
    Document(
        page_content="The stock market fell 500 points today due to fears of an impending economic slowdown.",
        metadata={"source": "news", "category": "finance", "author": "financial_gazette"},
    ),
    Document(
        page_content="Pinecone serverless indexes decouple storage from compute, eliminating manual cluster management.",
        metadata={"source": "website", "category": "tech", "author": "cloud_digest"},
    ),
]

print(f"Created {len(documents)} structured Document objects.")

In [ ]:
# Step 6: Ingest documents into the Pinecone Vector Store
# Embeddings (384d) are computed and indexed alongside metadata tags
inserted_ids = vector_store.add_documents(documents=documents)

print(f"Successfully ingested {len(inserted_ids)} documents into the vector store.")

In [ ]:
# Step 7: Basic Similarity Search
query = "What is the forecast for tomorrow's weather?"
results = vector_store.similarity_search(query, k=2)

print(f"Query: '{query}'")
print("=" * 60)
for i, res in enumerate(results, 1):
    print(f"Result {i}:")
    print(f"Content:  {res.page_content}")
    print(f"Metadata: {res.metadata}\n")

In [ ]:
# Step 8: Metadata Filtering Query
# Pinecone provides single-stage metadata filtering:
# Filtering occurs concurrently with index traversal, avoiding post-filtering recall loss.
query = "LangChain and Pinecone provide abstractions to make working with LLMs easy"

# Search specifically for tweets
results = vector_store.similarity_search(
    query,
    k=2,
    filter={"source": "tweet"}
)

print(f"Filtered Search (source=tweet) for query: '{query}'\n")
for res in results:
    print(f'* "{res.page_content}" | Metadata: {res.metadata}')

In [ ]:
# Step 9: Similarity Search with Numerical Scores
query = "Will it be hot and sunny tomorrow?"
results_with_score = vector_store.similarity_search_with_score(
    query,
    k=2,
    filter={"source": "news"}
)

print(f"Similarity Search with Scores (source=news): '{query}'\n")
for res, score in results_with_score:
    print(f'* [Score={score:.4f}] "{res.page_content}" | Metadata: {res.metadata}')

In [ ]:
# Step 10: Convert Pinecone Vector Store to a LangChain Retriever
# Using similarity score threshold filtering
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 2, "score_threshold": 0.3},
)

test_docs = retriever.invoke("Robbery and criminal activity at the bank")
print(f"Retriever returned {len(test_docs)} documents above score threshold:")
for doc in test_docs:
    print(f"- {doc.page_content} (Source: {doc.metadata['source']})")

In [ ]:
# Step 11: Build an End-to-End RAG Chain with Groq LLM & LCEL
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Document formatter helper
def format_docs(docs):
    return "\n\n".join(f"[Source: {doc.metadata.get('source')}]\n{doc.page_content}" for doc in docs)

# Grounded QA Prompt
pinecone_prompt = ChatPromptTemplate.from_template("""You are an enterprise AI assistant powered by Pinecone Serverless and Groq.
Answer the question using ONLY the provided context snippets.
If the information is not in the context, clearly explain that you do not know.

Context:
{context}

Question:
{question}

Answer:""")

# Build the LCEL Chain
pinecone_rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | pinecone_prompt
    | llm
    | StrOutputParser()
)

print("Pinecone + Groq LCEL RAG Chain constructed successfully!")

# Run test invocation
test_q = "What framework is best for stateful agentic workflows?"
print(f"\nQuery: {test_q}")
print("=" * 60)
answer = pinecone_rag_chain.invoke(test_q)
print(f"Groq LLM Answer:\n{answer}")

In [ ]:
# Step 12: Advanced Conversational RAG with Multi-Turn Memory
try:
    from langchain.chains import create_history_aware_retriever, create_retrieval_chain
    from langchain.chains.combine_documents import create_stuff_documents_chain
except ImportError:
    from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# Step 12a: Reformulation prompt to handle conversational context and pronouns
reformulate_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given chat history and the latest user question, rephrase it into a standalone question without chat history dependencies. Do NOT answer it."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(llm, retriever, reformulate_prompt)

# Step 12b: QA prompt incorporating chat history
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using the following retrieved context. Keep answers concise.\n\nContext:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

doc_chain = create_stuff_documents_chain(llm, qa_prompt)
conversational_rag = create_retrieval_chain(history_aware_retriever, doc_chain)

# Multi-turn demo
chat_history = []
print("=== TURN 1 ===")
q1 = "What happened to the stock market?"
print(f"User: {q1}")
res1 = conversational_rag.invoke({"chat_history": chat_history, "input": q1})
print(f"AI: {res1['answer']}\n")

chat_history.extend([HumanMessage(content=q1), AIMessage(content=res1['answer'])])

print("=== TURN 2 (Follow-up) ===")
q2 = "Why did it drop?"  # 'it' refers to the stock market from Turn 1
print(f"User: {q2}")
res2 = conversational_rag.invoke({"chat_history": chat_history, "input": q2})
print(f"AI: {res2['answer']}")